# Lab 4 — Build the Audience, Then Buy It
Name: Volkov Oleksii | Group: KM-42

## Setup

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


## Task 1.0 — Clean the survey file

In [2]:
va = viewer_attributes                               # age (300) exists only for R and L people
people = pd.DataFrame({"age": va[va.attr_id == 300].set_index("respondent_id").value,
                       "sex": va[va.attr_id.isin([201, 202])].set_index("respondent_id").attr_id.map({201: "M", 202: "W"})})
adult = people.age >= 18

raw = survey[(survey.RESPID != "RESPID") & ~survey.RESPID.str.startswith("TOTAL")]   # repeated headers, trailer
s = pd.DataFrame({"id": pd.to_numeric(raw.RESPID.str.strip()).astype("int64"),
                  "kw": raw.KEYWORD.str.strip().str.upper(),
                  "yes": raw.VALUE.str.strip().str.lower().isin(["1", "1.0", "y", "yes", "true"])})
codes = survey_layout.KEYWORD[survey_layout.RECTYPE == "B"]
ok = s.yes & s.kw.isin(codes) & s.id.isin(people.index[adult])
dups = s[ok].duplicated(["id", "kw"]).sum()
w = pd.crosstab(s.id[ok], s.kw[ok]).gt(0).reindex(people.index, fill_value=False)   # person x answer, yes/no
pc = ["X710101", "X710102", "X710103"]                # 1, 2, 3+ pairs
two = w[pc].sum(axis=1) > 1
w["X710101"] &= ~w.X710102 & ~w.X710103               # two pair counts: keep the higher one
w["X710102"] &= ~w.X710103
no_any = w.any(axis=1) & ~w.X710100
w["X710100"] = w.any(axis=1)                          # a pair count or any brand = buyer
buyers = pd.DataFrame({"buyer": w.X710100, "pairs": w[pc].dot([1, 2, 3]).astype(str).replace("3", "3+"),
                       "stride": w.X710201, "kestrel": w.X710202, "easywalk": w.X710203})[adult]
print("Survey data rows:", len(raw))
print("Clean yes answers:", ok.sum() - dups)
print("Adult panel members:", len(buyers))
print("Non-buyers:", (~buyers.buyer).sum())
print("People with two pair counts:", two.sum())
r, k = raw.RESPID, raw.KEYWORD
display(pd.DataFrame([
    ("repeated header line / trailer line", len(survey) - len(raw), "dropped"),
    ("RESPID with spaces (' 10070002')", (r != r.str.strip()).sum(), "stripped"),
    ("RESPID as a float (10088901.0)", r.str.contains(".", regex=False).sum(), "read as an integer"),
    ("RESPID with a leading zero (010391301)", r.str.strip().str.startswith("0").sum(), "read as an integer"),
    ("KEYWORD with spaces ('X710298 ')", (k != k.str.strip()).sum(), "stripped"),
    ("KEYWORD in lower case (x710100)", k.str.contains("x").sum(), "upper-cased"),
    ("yes written as 1.0 / y / Y / yes / TRUE", (s.yes & (raw.VALUE != "1")).sum(), "read as a yes"),
    ("VALUE not a yes (0, 0.0, N, no, empty)", (~s.yes).sum(), "dropped"),
    ("KEYWORD not in the layout (X880..., X902...)", (~s.kw.isin(codes)).sum(), "dropped"),
    ("RESPID of a child (under 18)", s.id.map(people.age).lt(18).sum(), "dropped"),
    ("RESPID of a guest (...81, ...82)", (s.id % 100 > 80).sum(), "dropped"),
    ("RESPID not in the panel (12345678, homes 104257+)", (~s.id.isin(va.respondent_id) & (s.id % 100 <= 80)).sum(), "dropped"),
    ("same person and keyword twice", dups, "dropped"),
    ("two pair counts (people)", two.sum(), "kept the higher one"),
    ("pair count or brand but no Any row (people)", no_any.sum(), "counted as a buyer"),
], columns=["bad record", "rows", "what I did"]))
lay = survey_layout[survey_layout.RECTYPE == "B"].set_index("KEYWORD")
display(pd.DataFrame({"Text": lay.Text, "my count": w[adult].sum(), "layout SAMPLE": lay.SAMPLE.astype(int)}))
display(buyers.head(3))                               # one row per adult panel member
print("Kestrel buyers, raw file vs clean:", (survey.KEYWORD == "X710202").sum(), buyers.kestrel.sum())
print("Share of buyers, raw file vs clean:", round((survey.KEYWORD == "X710100").sum() / len(buyers), 3), round(buyers.buyer.mean(), 3))

Survey data rows: 18346
Clean yes answers: 15510
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,bad record,rows,what I did
0,repeated header line / trailer line,3,dropped
1,RESPID with spaces (' 10070002'),573,stripped
2,RESPID as a float (10088901.0),533,read as an integer
3,RESPID with a leading zero (010391301),349,read as an integer
4,KEYWORD with spaces ('X710298 '),370,stripped
5,KEYWORD in lower case (x710100),734,upper-cased
6,yes written as 1.0 / y / Y / yes / TRUE,1194,read as a yes
7,"VALUE not a yes (0, 0.0, N, no, empty)",742,dropped
8,"KEYWORD not in the layout (X880..., X902...)",738,dropped
9,RESPID of a child (under 18),273,dropped


,Text,my count,layout SAMPLE
X710100,Any,4779,4779
X710101,1 pair,1810,1810
X710102,2 pairs,1308,1308
X710103,3 or more pairs,1661,1661
X710201,STRIDE,1563,1563
X710202,KESTREL,354,354
X710203,EASYWALK,1050,1050
X710298,OTHER BRAND,2739,2739
X710299,DON'T KNOW,371,371


,buyer,pairs,stride,kestrel,easywalk
respondent_id,,,,,
10000101,True,3+,False,False,False
10000102,True,2,False,False,True
10000201,True,1,True,False,False


Kestrel buyers, raw file vs clean: 516 354
Share of buyers, raw file vs clean: 0.572 0.597


**Answer:** The file broke all three promises: 742 rows are not a yes, 738 codes are not in the layout, 423 ids are not adult panelists (273 children, 84 guests, 66 not in the panel) and 933 rows are duplicates. It also has messy formats (padded, `.0` and zero-led ids, lower-case codes, *yes* spelled 6 ways), 3 header/trailer lines, 113 people with two pair counts and 238 buyers without an *Any* row. After cleaning, every code matches SAMPLE. Used as delivered, it would give 516 Kestrel buyers instead of 354 (+46%) and a buyer share of 0.572 instead of 0.597 (−2.5 points), so it is off in both directions.

## Task 1.1 — Four definitions

In [3]:
aud = pd.DataFrame({"P2+": True, "Adults 18+": adult, "Shoe buyers": w.X710100, "Heavy buyers": w.X710103,
                    "Kestrel buyers": w.X710202, "Adults 25-54": people.age.between(25, 54)})
pdays = daily_weights.merge(people.join(aud), left_on="respondent_id", right_index=True)   # R and L person-days
size = pdays[aud.columns].mul(pdays.weight, axis=0).groupby(pdays.broadcast_date).sum()   # daily audience sizes
U = size.mean()                                       # audience size = mean of the 31 daily sums
print("Adults 18+ (M):", round(U["Adults 18+"] / 1e6, 1))
print("Shoe buyers (M):", round(U["Shoe buyers"] / 1e6, 1))
print("Heavy buyers, 3+ pairs (M):", round(U["Heavy buyers"] / 1e6, 1))
print("Kestrel buyers (M):", round(U["Kestrel buyers"] / 1e6, 1))
print("Adults 25-54 (M):", round(U["Adults 25-54"] / 1e6, 1))
print("Kestrel buyers in panel (n):", buyers.kestrel.sum())

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [4]:
a = pdays[pdays.age >= 18]
group = a.sex + pd.cut(a.age, [17, 34, 54, 200], labels=["18-34", "35-54", "55+"]).astype(str)
prof = a[aud.columns[1:]].mul(a.weight, axis=0).groupby(group).sum()
share = prof / prof.sum()
index = share.div(share["Adults 18+"], axis=0) * 100
pd.concat({"share": share.round(3), "index vs adults": index.round(1)}, axis=1)

share                                                       \
       Adults 18+ Shoe buyers Heavy buyers Kestrel buyers Adults 25-54   
M18-34      0.144       0.140        0.111          0.118        0.174   
M35-54      0.160       0.158        0.138          0.189        0.326   
M55+        0.185       0.186        0.189          0.126        0.000   
W18-34      0.144       0.140        0.139          0.181        0.174   
W35-54      0.160       0.178        0.226          0.265        0.326   
W55+        0.208       0.197        0.196          0.121        0.000   

       index vs adults                                                       
            Adults 18+ Shoe buyers Heavy buyers Kestrel buyers Adults 25-54  
M18-34           100.0        97.4         77.1           82.1        121.2  
M35-54           100.0        99.2         86.8          118.6        204.1  
M55+             100.0       100.8        102.2           68.2          0.0  
W18-34           100.0        97.6         97.0          126.0        121.2  
W35-54           100.0       111.4        141.9          165.8        204.1  
W55+             100.0        94.6         94.0           58.0          0.0

**Answer:** I would hand over heavy buyers (3+ pairs): it is the brief itself, it is measurable (49.6M people, 1,661 panelists) and it differs from adults (women 35–54: index 142). Kestrel buyers are only 354 panelists, and adults 25–54 leaves out the 55+, who are 38.5% of heavy buyers. Shoe buyers look like adults because they are 59% of adults, and a group that is most of the population cannot differ much from it (indexes 95–111). A fused attribute has a limit: purchases are assigned by demographics, so it cannot show whether a program's viewers buy more than people of the same age, sex and income who do not watch it.

## Task 2.1 — Programs by audience

In [5]:
pv = viewings[viewings.resp_type.isin(["R", "L"])].merge(aud, left_on="respondent_id", right_index=True)
pv["broadcast_date"] = (pv.start_ts - pd.Timedelta(hours=6)).dt.date

def ratings(tel, cols):
    """Average-minute rating: sum(weight x minutes viewed) / (telecast minutes x that day's audience size) x 100."""
    x = tel.merge(pv, on=["network_code", "broadcast_date"], suffixes=("", "_v"))
    mins = (x[["end_ts", "end_ts_v"]].min(axis=1) - x[["start_ts", "start_ts_v"]].max(axis=1)).dt.total_seconds().clip(lower=0) / 60
    num = x[cols].mul(x.weight * mins, axis=0).groupby(x.telecast_id).sum()
    t = tel.set_index("telecast_id")
    return num.reindex(t.index, fill_value=0) / size.loc[t.broadcast_date, cols].values / t[["duration_min"]].values * 100

def cost30(tel):
    """Rate-card price of a :30 in each telecast (Lab 2): program rows win over time rows."""
    prog = tel.merge(rate_card, on=["network_code", "program_name"]).set_index("telecast_id").cost_30s_usd
    x = tel.merge(rate_card[rate_card.program_name.isna()].explode("weekdays"), on="network_code", suffixes=("", "_rc"))
    day = pd.to_datetime(x.broadcast_date)
    clock = x.start_ts - day                          # broadcast clock: 25:37 = 01:37 next morning
    x = x[(x.weekdays == day.dt.weekday) & (pd.to_timedelta(x.start_time + ":00") <= clock) & (clock < pd.to_timedelta(x.end_time + ":00"))]
    return prog.combine_first(x.set_index("telecast_id").cost_30s_usd)

eight = pd.DataFrame([("Sunday Night Football (NBC)", "NBC", "NFL Football"), ("Monday Night Football (ABC)", "ABC", "NFL Football"),
                      ("Dancing with the Stars (ABC)", "ABC", "Dancing with the Stars"), ("NCIS (CBS)", "CBS", "NCIS"),
                      ("Hannity (FNC)", "FNC", "Hannity"), ("Unsellable Houses (HGTV)", "HGTV", "Unsellable Houses"),
                      ("Halloween Wars (FOOD)", "FOOD", "Halloween Wars"), ("Grotesquerie (FX)", "FX", "Grotesquerie")],
                     columns=["program", "network_code", "program_name"])
tel = eight.merge(programs[programs.daypart == "Prime"])          # October prime-time telecasts
tel = tel.join(ratings(tel, ["Adults 18+", "Heavy buyers", "Kestrel buyers"]), on="telecast_id")
tel["cost"] = tel.telecast_id.map(cost30(tel))
m = tel.groupby("program", sort=False)[["Adults 18+", "Heavy buyers", "Kestrel buyers", "cost"]].mean()
adults000 = m["Adults 18+"] * U["Adults 18+"] / 1e5   # viewers in an average minute (000) = rating / 100 x size / 1000
t2 = pd.DataFrame({"adults rating": m["Adults 18+"], "target rating": m["Heavy buyers"],
                   "target viewers (000)": m["Heavy buyers"] * U["Heavy buyers"] / 1e5})
t2["composition"] = t2["target viewers (000)"] / adults000
t2["index"] = t2["target rating"] / t2["adults rating"] * 100
t2["cost per :30"] = m.cost
t2["adults CPM"] = m.cost / adults000
t2["target CPM"] = m.cost / t2["target viewers (000)"]
display(t2.round({"adults rating": 2, "target rating": 2, "target viewers (000)": 0, "composition": 3, "index": 1,
                  "cost per :30": 0, "adults CPM": 0, "target CPM": 0}))
print("Target universe (M):", round(U["Heavy buyers"] / 1e6, 1))

,adults rating,target rating,target viewers (000),composition,index,cost per :30,adults CPM,target CPM
program,,,,,,,,
Sunday Night Football (NBC),5.82,5.79,2869.0,0.195,99.3,487200.0,33.0,170.0
Monday Night Football (ABC),8.93,8.36,4143.0,0.184,93.6,723600.0,32.0,175.0
Dancing with the Stars (ABC),3.25,3.14,1559.0,0.190,96.8,158600.0,19.0,102.0
NCIS (CBS),2.21,2.43,1204.0,0.216,110.1,178800.0,32.0,148.0
Hannity (FNC),1.09,0.84,417.0,0.151,77.1,39900.0,15.0,96.0
Unsellable Houses (HGTV),0.50,0.64,317.0,0.249,126.8,12900.0,10.0,41.0
Halloween Wars (FOOD),0.28,0.33,162.0,0.228,116.2,9700.0,14.0,60.0
Grotesquerie (FX),0.26,0.25,124.0,0.188,95.9,7592.0,11.0,61.0


Target universe (M): 49.6


## Task 2.2 — Index, volume or price?

In [6]:
print("Highest index (program):", t2["index"].idxmax())
print("Most target viewers (program):", t2["target viewers (000)"].idxmax())
print("Lowest target CPM (program):", t2["target CPM"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** No. Unsellable Houses (HGTV) has the highest index (127) and the lowest target CPM (\$41), but Monday Night Football has the most heavy buyers: 4.14M in an average minute against 0.32M, 13 times as many. I would tell the client that an index measures concentration, not size. An HGTV spot reaches 0.6% of the target, and 75% of its viewers are still not heavy buyers, so buying only the top index repeats the same small group. Index finds efficiency and coverage finds reach (Lecture 5's shampoo example), so the plan needs both, compared by target CPM.

## Task 2.3 — Kestrel buyers

In [7]:
n = pdays.groupby("broadcast_date")[["Heavy buyers", "Kestrel buyers"]].sum().mean()   # in-tab panelists per day
seen = m[["Heavy buyers", "Kestrel buyers"]] / 100 * n             # target panelists behind an average minute
pd.DataFrame({"heavy-buyer index": t2["index"], "Kestrel index": m["Kestrel buyers"] / m["Adults 18+"] * 100,
              "heavy panelists per minute": seen["Heavy buyers"], "Kestrel panelists per minute": seen["Kestrel buyers"],
              "heavy ±error (%)": 100 / seen["Heavy buyers"] ** 0.5, "Kestrel ±error (%)": 100 / seen["Kestrel buyers"] ** 0.5}).round(1)

,heavy-buyer index,Kestrel index,heavy panelists per minute,Kestrel panelists per minute,heavy ±error (%),Kestrel ±error (%)
program,,,,,,
Sunday Night Football (NBC),99.3,98.5,86.4,18.3,10.8,23.4
Monday Night Football (ABC),93.6,85.9,124.8,24.4,9.0,20.2
Dancing with the Stars (ABC),96.8,82.7,47.0,8.6,14.6,34.2
NCIS (CBS),110.1,56.5,36.3,4.0,16.6,50.2
Hannity (FNC),77.1,42.6,12.5,1.5,28.2,82.2
Unsellable Houses (HGTV),126.8,76.9,9.6,1.2,32.3,89.9
Halloween Wars (FOOD),116.2,95.2,4.9,0.9,45.3,108.3
Grotesquerie (FX),95.9,129.6,3.7,1.1,51.7,96.3


**Answer:** It jumps because there are only 354 Kestrel panelists, and 5 of the 8 programs have 4 or fewer of them in an average minute. With an error of about 1/√n (Lecture 6) that is ±50–110%, against ±9–52% for heavy buyers. I would trust the heavy-buyer index, above all for SNF, MNF, Dancing with the Stars and NCIS (±9–17%), and use the Kestrel index only as a hint. To check, count the panelists behind each index (table above), compute standard errors or bootstrap, and test whether the index holds across telecasts and weeks.

## Task 3.1 — Weights

In [8]:
start = acr_sessions.start_utc.dt.tz_convert("America/New_York").dt.tz_localize(None)   # UTC -> US Eastern
end = acr_sessions.end_utc.dt.tz_convert("America/New_York").dt.tz_localize(None)
acr = acr_sessions.assign(start=start, end=end, broadcast_date=(start - pd.Timedelta(hours=6)).dt.date)[end - start >= pd.Timedelta(minutes=1)]
hh1 = hh_attributes[hh_attributes.groupby("hashed_ip").household_id.transform("nunique") == 1]   # IP of exactly one home
tvs = acr_sessions[["tv_id", "hashed_ip"]].drop_duplicates().merge(hh1, on="hashed_ip")
tvs["head_age"] = tvs.head_age_band.eq("65+").map({True: "65+", False: "Under 65"})
tab = acr[["tv_id", "broadcast_date"]].drop_duplicates().merge(tvs[["tv_id", "head_age", "income_band"]]).merge(universe)
tab["weight"] = tab.tv_households / tab.groupby(["broadcast_date", "head_age", "income_band"]).tv_id.transform("size")
print("TVs matched to one household:", len(tvs))
print("Weights sum per day (M):", tab.groupby("broadcast_date").weight.sum().div(1e6).round(1).unique())
cell = tab.groupby(["head_age", "income_band"]).agg(tv_days=("tv_id", "size"), tv_households=("tv_households", "first"))
display((cell.tv_days / cell.tv_days.sum() / (cell.tv_households / cell.tv_households.sum()) * 100).unstack().round(1))  # footprint index

ref = reference_ratings.merge(programs, on="telecast_id")
ref["telecast"] = ref.program_name + " (" + ref.network_code + "), " + ref.broadcast_date.astype(str)
x = acr[acr.session_type == "live"].merge(ref, on=["network_code", "broadcast_date"])
x["sec"] = (x[["end", "end_ts"]].min(axis=1) - x[["start", "start_ts"]].max(axis=1)).dt.total_seconds().clip(lower=0)
x = x.groupby(["telecast_id", "tv_id"]).sec.sum().loc[lambda v: v >= 60].reset_index()   # TVs that watched >= 1 minute
g = ref[["telecast", "telecast_id", "broadcast_date"]].merge(tab).merge(x, how="left", indicator=True)   # matched in-tab TVs
g["watched"] = g._merge == "both"
by = g.groupby("telecast", sort=False)
t3 = pd.DataFrame({"unweighted": by.watched.mean() * 100,
                   "weighted": (g.watched * g.weight).groupby(g.telecast, sort=False).sum() / by.weight.sum() * 100,
                   "reference": ref.set_index("telecast").hh_rating})
display(t3.round(2))
display(pd.concat([g.pivot_table("watched", "telecast", c) * 100 for c in ["head_age", "income_band"]], axis=1).round(2))  # unweighted
gap_u, gap_w = (t3.unweighted - t3.reference).abs().mean(), (t3.weighted - t3.reference).abs().mean()
print("Mean gap unweighted (points):", round(gap_u, 2))
print("Mean gap weighted (points):", round(gap_w, 2))
print("Share of gap closed:", round(1 - gap_w / gap_u, 3))

TVs matched to one household: 19607
Weights sum per day (M): [125.8]


income_band,$100k-$150k,$150k+,$25k-$50k,$50k-$75k,$75k-$100k,Under $25k
head_age,,,,,,
65+,90.9,93.0,54.0,64.5,85.6,46.2
Under 65,127.1,117.5,88.1,105.9,137.8,86.0


,unweighted,weighted,reference
telecast,,,
"MLB Postseason (FOX), 2024-10-25",17.03,17.76,25.86
"NFL Football (NBC), 2024-10-27",22.62,23.75,33.14
"NCIS (CBS), 2024-10-14",5.50,5.80,10.56
"Hannity (FNC), 2024-10-16",1.04,1.35,2.30
"Unsellable Houses (HGTV), 2024-10-16",2.12,2.08,3.69
"Halloween Baking Championship (FOOD), 2024-10-14",0.95,0.90,1.63
"Grotesquerie (FX), 2024-10-16",2.17,2.11,3.53
"The Real Housewives of New York City (BRAVO), 2024-10-15",0.71,0.64,1.23


,65+,Under 65,$100k-$150k,$150k+,$25k-$50k,$50k-$75k,$75k-$100k,Under $25k
telecast,,,,,,,,
"Grotesquerie (FX), 2024-10-16",1.97,2.22,2.34,2.75,2.14,1.61,1.99,1.38
"Halloween Baking Championship (FOOD), 2024-10-14",0.97,0.94,0.99,1.21,0.84,0.60,1.14,0.51
"Hannity (FNC), 2024-10-16",3.49,0.44,1.01,0.77,1.34,1.09,0.95,1.48
"MLB Postseason (FOX), 2024-10-25",25.32,15.04,16.90,19.43,15.46,16.45,16.01,15.32
"NCIS (CBS), 2024-10-14",8.59,4.76,5.50,5.89,4.49,5.95,5.38,5.26
"NFL Football (NBC), 2024-10-27",34.93,19.65,23.46,24.03,20.96,22.68,21.42,20.89
"The Real Housewives of New York City (BRAVO), 2024-10-15",0.34,0.80,0.84,1.14,0.46,0.60,0.33,0.36
"Unsellable Houses (HGTV), 2024-10-16",2.34,2.07,2.31,2.46,1.51,2.17,2.05,1.58


Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075


## Task 3.2 — What weighting cannot fix
**Answer:** Weighting moved the ratings because 65+ homes are under-represented (cell indexes 46–93) and get more weight. Older-skewing telecasts rose (Hannity 1.04 → 1.35, NCIS, SNF, World Series) and younger ones dipped (Bravo, FX, Food, HGTV: −0.04 to −0.07). Yet only 7.5% of the gap closed, and weighted ratings remain 28–48% below the reference. Weighting fixes who is in the sample, not what each TV sees: ACR covers about one of ~2.6 TVs per home (Lecture 4), unrecognised viewing is lost, only 65% of TVs match a home, and opt-in homes may differ from others in the same cell. Income barely matters because viewing hardly differs by income (SNF 21–24% and NCIS 4.5–6.0% in every band).

## Task 4.1 — October predicts October

In [9]:
big = programs[(programs.daypart == "Prime") & programs.network_code.isin(["CBS", "NBC", "ABC", "FOX", "FNC"])]
big = big.join(ratings(big, ["P2+", "Heavy buyers"]), on="telecast_id")
d = pd.to_datetime(big.broadcast_date)
history = big[d <= "2024-10-21"].groupby(["network_code", "program_name"])[["P2+", "Heavy buyers"]].mean()
fc = big[(d >= "2024-10-22") & (d <= "2024-10-28")].join(history, on=["network_code", "program_name"], rsuffix=" forecast", how="inner")
for c in ["P2+", "Heavy buyers"]:
    fc[c + " error"] = (fc[c] - fc[c + " forecast"]).abs() / fc[c].where(fc[c] > 0)   # actual 0 -> NaN, skipped
fc["program"] = fc.program_name + " (" + fc.network_code + ")"
miss = fc.loc[fc["P2+ error"].idxmax()]
print("Telecasts forecast (n):", len(fc))
print("MAPE, P2+ (%):", round(fc["P2+ error"].mean() * 100, 1))
print("MAPE, heavy buyers (%):", round(fc["Heavy buyers error"].mean() * 100, 1))
print("Biggest miss, P2+ (program, date):", f"{miss.program}, {miss.broadcast_date}")
for c in ["P2+", "Heavy buyers"]:
    print(f"WAPE, {c} (%):", round((fc[c] - fc[c + " forecast"]).abs().sum() / fc[c].sum() * 100, 1),
          "| whole week, forecast vs actual (%):", round((fc[c + " forecast"].sum() / fc[c].sum() - 1) * 100, 1))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27
WAPE, P2+ (%): 22.8 | whole week, forecast vs actual (%): 2.6
WAPE, Heavy buyers (%): 24.6 | whole week, forecast vs actual (%): -0.2


## Task 4.2 — Why it missed

In [10]:
top = fc.nlargest(5, "P2+ error")
display(pd.DataFrame({"program": top.program, "date": top.broadcast_date, "time": top.start_ts.dt.strftime("%a %H:%M-") + top.end_ts.dt.strftime("%H:%M"),
                      "repeat": top.is_repeat, "forecast": top["P2+ forecast"], "actual": top["P2+"], "error (%)": top["P2+ error"] * 100}).round({"forecast": 2, "actual": 2, "error (%)": 1}))
h = big[d <= "2024-10-21"].merge(top[["network_code", "program_name"]]).sort_values(["program_name", "start_ts"])   # history behind them
pd.DataFrame({"program": h.program_name + " (" + h.network_code + ")", "date": h.broadcast_date,
              "time": h.start_ts.dt.strftime("%a %H:%M"), "repeat": h.is_repeat, "P2+ rating": h["P2+"].round(2)})

,program,date,time,repeat,forecast,actual,error (%)
3068,Krapopolis (FOX),2024-10-27,Sun 21:30-22:00,False,1.06,0.22,387.8
3066,Universal Basic Guys (FOX),2024-10-27,Sun 20:30-21:00,False,0.68,0.26,162.2
1544,Fire Country (CBS),2024-10-27,Sun 22:44-23:00,True,1.70,0.67,154.8
3037,Accused (FOX),2024-10-22,Tue 21:00-22:00,False,0.95,0.37,154.6
312,What Would You Do? (ABC),2024-10-23,Wed 22:00-23:00,False,0.44,0.18,138.1


,program,date,time,repeat,P2+ rating
4,Accused (FOX),2024-10-01,Tue 21:00,True,1.87
8,Accused (FOX),2024-10-08,Tue 21:00,False,0.61
10,Accused (FOX),2024-10-15,Tue 21:00,False,0.38
2,Fire Country (CBS),2024-10-18,Fri 21:00,False,1.81
3,Fire Country (CBS),2024-10-19,Sat 21:00,True,1.58
5,Krapopolis (FOX),2024-10-02,Wed 22:00,True,1.76
7,Krapopolis (FOX),2024-10-06,Sun 21:30,False,0.26
9,Krapopolis (FOX),2024-10-09,Wed 22:00,True,1.93
12,Krapopolis (FOX),2024-10-20,Sun 21:30,False,0.30
6,Universal Basic Guys (FOX),2024-10-06,Sun 20:30,False,0.26


**Answer:** All five are over-forecasts, and three fall on Sunday, Oct 27. CBS's NFL game ran to 19:35, which left Fire Country as a 16-minute repeat tail against SNF. The other forecasts average unlike airings: Universal Basic Guys' history includes Oct 20 at 20:00 after FOX's NFL game (1.09), but at 20:30 it gets 0.26; Krapopolis and Accused mix high repeats (1.76–1.93, 1.87) with originals (0.26–0.61), and the new originals got 0.22 and 0.37; What Would You Do? got 0.49 on a Monday after a two-hour Dancing with the Stars, but only 0.18 on Wednesday, Oct 23. The heavy-buyer forecast is less accurate (MAPE 38.6% against 35.8%) because heavy buyers are about a sixth of the panel, so sampling noise adds to the same schedule errors. A better forecast would use (1) schedule context: slot, day, original or repeat, lead-in, competition and overruns; and (2) for the target, a large-sample rating times a pooled, stable index (Lecture 6).

## Task 5.1 — The plan

In [11]:
spots = pd.Series({"Unsellable Houses (HGTV)": 6, "Grotesquerie (FX)": 6, "Hannity (FNC)": 10, "Dancing with the Stars (ABC)": 3})
plan = pd.DataFrame({"prime airings per week (Oct)": tel.program.value_counts()[spots.index] / 31 * 7,
                     "spots": spots, "cost": spots * t2["cost per :30"][spots.index],
                     "target impressions (000)": spots * t2["target viewers (000)"][spots.index],
                     "adult impressions (000)": spots * adults000[spots.index]})
display(plan.round(1))
print("Plan cost ($):", round(plan.cost.sum()))
print("Plan target impressions (M):", round(plan["target impressions (000)"].sum() / 1e3, 1))
print("Plan target CPM ($):", round(plan.cost.sum() / plan["target impressions (000)"].sum()))
print("Plan composition:", f'{plan["target impressions (000)"].sum() / plan["adult impressions (000)"].sum():.3f}')

,prime airings per week (Oct),spots,cost,target impressions (000),adult impressions (000)
Unsellable Houses (HGTV),2.9,6,77400.0,1904.5,7651.8
Grotesquerie (FX),5.6,6,45552.0,746.1,3962.3
Hannity (FNC),5.0,10,399000.0,4165.3,27513.4
Dancing with the Stars (ABC),2.0,3,475800.0,4678.2,24620.8


Plan cost ($): 997752
Plan target impressions (M): 11.5
Plan target CPM ($): 87
Plan composition: 0.180


## Task 5.2 — Recommendation
**Answer:** I would buy 25 :30s for \$997,752 (6 in Unsellable Houses, 6 in Grotesquerie, 10 in Hannity and 3 in Dancing with the Stars), giving 11.5M heavy-buyer impressions at a \$87 target CPM, about half the NFL's \$170–175. I chose by target CPM: HGTV is the cheapest and highest-index program, Dancing with the Stars adds volume, and the cable buys are capped by their 3–6 prime airings a week. I skipped the NFL (one MNF spot is 72% of the budget) and Halloween Wars, which is seasonal and off air in November. Delivery is uncertain: in Task 4, heavy-buyer forecasts missed single telecasts by 39% (MAPE), so I would plan for ±25% (the WAPE) and ask for an audience guarantee with make-goods. Composition is 0.180, so about 82% of the impressions, and of the money, reach people who are not heavy buyers (85% on Hannity). After the week I would compare delivered heavy-buyer impressions per spot with the plan, along with reach, frequency and, if purchases can be matched, sales lift.

## Bonus

In [18]:
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print(kagglehub.whoami())

Kaggle credentials successfully validated.
{'username': 'aleksi0lyoshka'}


In [19]:
import numpy as np
import lightgbm as lgb
TEST = kagglehub.competition_download("forecast-november-tv-ratings-by-audience") + "/test.csv"
A5 = ["P2+", "A18-49", "W25-54", "SHOE", "SHOE3"]
aud5 = pd.DataFrame({"P2+": True, "A18-49": people.age.between(18, 49), "W25-54": (people.sex == "W") & people.age.between(25, 54),
                     "SHOE": w.X710100, "SHOE3": w.X710103})
d5 = daily_weights.merge(aud5, left_on="respondent_id", right_index=True)
size5 = d5[A5].mul(d5.weight, axis=0).groupby(d5.broadcast_date).sum()

# October panel rating of every telecast for the five audiences (sessions joined to the telecasts they overlap)
v5 = viewings[viewings.resp_type.isin(["R", "L"])].merge(aud5, left_on="respondent_id", right_index=True)
num = []
for net, tp in programs.sort_values("start_ts").groupby("network_code"):
    vv = v5[v5.network_code == net]
    i0 = np.searchsorted(tp.end_ts.values, vv.start_ts.values, "right")
    i1 = np.searchsorted(tp.start_ts.values, vv.end_ts.values)
    k = np.repeat(np.arange(len(vv)), np.maximum(i1 - i0, 0))
    j = np.concatenate([np.arange(a, b) for a, b in zip(i0, i1)])
    mins = (np.minimum(vv.end_ts.values[k], tp.end_ts.values[j]) - np.maximum(vv.start_ts.values[k], tp.start_ts.values[j])) / np.timedelta64(1, "m")
    num.append(vv[A5].iloc[k].mul(vv.weight.values[k] * mins, axis=0).groupby(tp.telecast_id.values[j]).sum())
t = programs.set_index("telecast_id")
octr = (pd.concat(num).reindex(t.index, fill_value=0) / size5.loc[t.broadcast_date, A5].values / t[["duration_min"]].values * 100)
octr = t.join(octr).reset_index().melt(id_vars=list(t.reset_index().columns), value_vars=A5, var_name="audience", value_name="rating")

def feats(df):
    d, st = pd.to_datetime(df.broadcast_date), pd.to_datetime(df.start_ts)
    clock = (st - d).dt.total_seconds() / 3600                       # broadcast clock, 25.5 = 01:30
    return df.assign(dow=d.dt.weekday, wknd=d.dt.weekday.clip(4, 6), hour=clock // 1, half=clock * 2 // 1 / 2,
                     rep=df.is_repeat.astype(str).str.lower().eq("true").astype(int))

def mean_by(tr, te, keys, loo):                                     # group mean (leave-one-out for training rows)
    g = tr.groupby(keys + ["audience"]).rating.agg(["sum", "count"])
    j = te.join(g, on=keys + ["audience"])
    s, c = j["sum"].fillna(0).values, j["count"].fillna(0).values
    if loo:
        s, c = s - te.rating.values, c - 1
    return np.where(c > 0, s / np.maximum(c, 1), np.nan), np.maximum(c, 0)

def backoff(tr, te, levels, k, loo):                                # general -> specific, each level shrunk toward the last
    pred = None
    for keys in levels:
        m, c = mean_by(tr, te, keys, loo)
        if pred is None:
            pred = m
        else:
            has = ~np.isnan(pred)
            pred = np.where(c > 0, (np.nan_to_num(m) * c + k * np.nan_to_num(pred)) / np.maximum(c + k * has, 1e-9), pred)
    return np.nan_to_num(pred)

P, NET = ["network_code", "program_name"], [["network_code", "daypart"]]
ORIG = NET + [P, P + ["rep"], P + ["rep", "hour"], P + ["rep", "wknd", "hour"]]     # originals follow the program
REPS = NET + [["network_code", "wknd", "hour", "rep"], ["network_code", "dow", "half", "rep"]]   # repeats follow the slot
TE = {"prog": P, "prog_rep": P + ["rep"], "prog_rep_dp": P + ["rep", "daypart"], "prog_rep_hour": P + ["rep", "hour"],
      "slot": ["network_code", "wknd", "hour", "rep"], "slot_dow": ["network_code", "dow", "half", "rep"],
      "net_dp": ["network_code", "daypart"], "type": ["network_code", "program_type", "rep"]}
CAT = ["audience", "network_code", "daypart", "program_type"]

def forecast(tr, te, seeds=range(5), rounds=400):
    tr, te = feats(tr), feats(te)
    hier = lambda x, loo: np.where(x.rep == 1, backoff(tr, x, REPS, 2, loo), backoff(tr, x, ORIG, 1, loo))
    def X(x, ref, loo):
        out = pd.DataFrame({n: mean_by(tr, x, keys, loo)[0] / ref for n, keys in TE.items()}, index=x.index)
        out[["ref", "rep", "dur", "hour", "dow"]] = np.c_[ref, x.rep, x.duration_min, x.hour, x.dow]
        for c in CAT:
            out[c] = pd.Categorical(x[c].astype(str), categories=sorted(set(tr[c].astype(str)) | set(te[c].astype(str))))
        return out
    ref_tr, ref_te = np.maximum(hier(tr, True), 0.01), np.maximum(hier(te, False), 0.01)
    prm = dict(objective="l1", learning_rate=0.03, num_leaves=31, min_data_in_leaf=20, feature_fraction=0.8,
               bagging_fraction=0.8, bagging_freq=1, verbose=-1)
    data = lgb.Dataset(X(tr, ref_tr, True), tr.rating.values / ref_tr, weight=ref_tr)   # learn a multiplier, L1 = WAPE
    mult = np.mean([lgb.train({**prm, "seed": s}, data, rounds).predict(X(te, ref_te, False)) for s in seeds], axis=0)
    gbm = np.clip(mult, 0, None) * ref_te
    return np.where(ref_te > 2, (gbm + ref_te) / 2, gbm)            # big telecasts: lean on the hierarchy

wape = lambda y, f: np.abs(y - f).sum() / y.sum() * 100
dd = pd.to_datetime(octr.broadcast_date)
early, late = octr[dd <= "2024-10-17"], octr[dd > "2024-10-17"]    # check: Oct 1-17 -> Oct 18-31
base = mean_by(early, late, P, False)[0]                                  # "same as October", new programs -> network x daypart
base = np.where(np.isnan(base), mean_by(early, late, ["network_code", "daypart"], False)[0], base)
print("Validation WAPE, same as October:", round(wape(late.rating.values, base), 1))
print("Validation WAPE, model:", round(wape(late.rating.values, forecast(early, late)), 1))

test = pd.read_csv(TEST)
test["rating"] = forecast(octr, test.assign(rating=np.nan))
# Election night: published Nielsen viewers (M, P2+, 8-11 pm ET), scaled to this panel's level with the VP debate
debate = {"ABC": 6.35, "CBS": 9.57, "NBC": 5.67, "FNC": 7.93}          # Oct 1, 2024 (Nielsen)
election = {"FNC": 10.32, "ABC": 5.90, "NBC": 5.51, "CBS": 3.6, "FOX": 2.0}   # Nov 5, 2024 (Nielsen)
deb = octr[octr.program_name.str.startswith("Vice Presidential Debate") & (octr.audience == "P2+")].set_index("network_code").rating
scale = (deb[list(debate)] / 100 * size5["P2+"].mean() / 1e6).sum() / sum(debate.values())
night = (test.broadcast_date == "2024-11-05") & test.program_name.str.contains("News Special Report|Democracy 2024")
p2 = test[night & (test.audience == "P2+")].set_index("telecast_id")
event = p2.network_code.map(election) * scale / size5["P2+"].mean() * 1e8          # published level as a rating
factor = ((p2.rating + event) / 2 / p2.rating)                                   # hedge: halfway to the event level
test.loc[night, "rating"] *= test.loc[night, "telecast_id"].map(factor).values
test[["id", "rating"]].to_csv("submission.csv", index=False)
print("Debate scale (panel / Nielsen):", round(scale, 2))
print("Election-night P2+ after the hedge:", (p2.rating * factor).round(2).set_axis(p2.network_code).to_dict())
print("Submission rows:", len(test), "| written to submission.csv")
print("Kaggle username:", "aleksi0lyoshka")

100%|██████████| 138k/138k [00:00<00:00, 43.5MB/s]

Extracting files...


Validation WAPE, same as October: 37.2
Validation WAPE, model: 22.9
Debate scale (panel / Nielsen): 0.89
Election-night P2+ after the hedge: {'ABC': 1.76, 'CBS': 1.55, 'FNC': 1.76, 'FOX': 1.19, 'NBC': 1.58}
Submission rows: 11570 | written to submission.csv
Kaggle username: aleksi0lyoshka


**Answer:** I rate every October telecast for the five audiences and forecast each November telecast from what drove October: originals from their own program (by repeat flag, hour and weekday), repeats from the network's time slot, and a LightGBM model with L1 loss (the WAPE numerator) that learns a correction factor on top; on Oct 1–17 → Oct 18–31 this cuts WAPE from 37.2 to 22.8. For Election Night I used public Nielsen viewer counts for the Oct 1 VP debate and the Nov 5 coverage, 8–11 pm (as reported by Nielsen, Axios, RBR and The Hollywood Reporter), scaled them to this panel with the debate and set each network's coverage halfway between the model and that level, because the October debate shows no event boost in this data.